# Cyndx AI Engineering Challenge: MMR Search Algorithm

At Cyndx we develop AI and algorithms that perform search, information retrieval, and other related AI services with our large database of private companies. "Search quality" is a hard-to-define metric that we think about a lot. One potential way to improve the perceived quality of search results is to reduce redundancy and increase diversity in the results returned to the user. Maximal Marginal Relevance (MMR) is a simple algorithm that accomplishes this goal.

## The Challenge

Implement a basic search algorithm that utilizes MMR to re-rank the search results.

### Requirements
1. Implement the MMR algorithm defined in the original paper: https://www.cs.cmu.edu/~jgc/publication/The_Use_MMR_Diversity_Based_LTMIR_1998.pdf
2. Utilize the provided dataset `./companies.csv` to demonstrate your implementation
3. The search should be composed of two parts:

    * A basic search algorithm with the signature `def search(query: str, ...) -> pd.DataFrame` that returns relevancy-based results for the given query
    * An MMR re-ranking algorithm with the signature `def rerank(results: pd.DataFrame, ...) -> pd.DataFrame`, returning the optimized results
    * The demonstration should compose these functions, for example: `rerank(search(query, ...), ...)`
4. After you have implemented and demonstrated your algorithms, please answer the questions at the bottom of this notebook
5. Please provide a `README.md` file with instructions on how to run your code, and a `requirements.txt` file with the necessary dependencies
6. Email us your solution as a .zip file containing your modified notebook and all necessary files

In [1]:
!pip3 install -r requirements.txt

You should consider upgrading via the '/Library/Frameworks/Python.framework/Versions/3.10/bin/python3.10 -m pip install --upgrade pip' command.


In [2]:
# Basic Python libraries
import os
import pandas as pd
import numpy as np

# Used-defined libraries
from UDFs import clean_text
from UDFs import generate_companies_embeddings
from UDFs import mmr

# Sentence Transformers
from sentence_transformers import SentenceTransformer

# sklearn libraries
from sklearn.metrics.pairwise import cosine_similarity

### Implementation

In [3]:
companies = pd.read_csv("./companies.csv")
companies.head()

,Name,EmployeeCount,Description,Url,Region,Country,MetroArea,City
0,Molan Steel Co.,11-50,Molan Steel Co. engages in supplying steel pro...,molansteel.com,RY,SA,Riyadh/Saudi Arabia Metro,Riyadh
1,Hunter Douglas NV,10001+,"Hunter Douglas NV engages in the design, manuf...",hunterdouglas.com,ZH,NL,Amsterdam/NL Metro,Rotterdam
2,"Root, Inc.",501-1000,"Root, Inc. is a technology insurance company, ...",inc.joinroot.com,OH,US,Columbus/OH Metro,Columbus
3,Bowlero Corp.,10001+,Bowlero Corp. engages in operating bowling cen...,bowlero.com,VA,US,Richmond/VA Metro,Mechanicsville
4,Meridia Real Estate III SOCIMI SA,1-10,Meridia Real Estate III SOCIMI SA operates as ...,meridiarealestateiiisocimi.com,CT,ES,Barcelona/Spain Metro,Barcelona


In [4]:
model = SentenceTransformer("all-MiniLM-L6-v2")

In [5]:
def search(query: str, df: pd.DataFrame, n=100) -> pd.DataFrame:
    # Clean the query
    query = clean_text(query)

    # Sentence Transformers can be used for more advanced search capabilities
    model = SentenceTransformer("all-MiniLM-L6-v2")

    # Generate embeddings for the companies
    df = generate_companies_embeddings(df, model)

    # Generate embeddings for the query
    query_embedding = model.encode([query])

    # Compute the cosine similarity between the query and the company embeddings
    df["similarity"] = df["embeddings"].apply(
        lambda x: cosine_similarity(query_embedding, [x])[0][0]
    )

    # Get the top n results based on similarity
    results = df.sort_values(by="similarity", ascending=False).reset_index(drop=True)
    results["query_embeddings"] = [query_embedding] * len(results)

    ### TODO: replace this with your own search algo:
    results = results.loc[:n, :].copy()

    return results

In [6]:
def rerank(results: pd.DataFrame, n=10) -> pd.DataFrame:

    # Query embeddings
    query_embeddings = results["query_embeddings"][0][0]

    # Extract document embeddings as a numpy array
    doc_embeddings = np.vstack(results["embeddings"].values)

    # Compute selected indices using MMR
    selected_indices = mmr(doc_embeddings, query_embeddings, n, 0.5)

    # Compute the MMR(Maximum Marginal Relevance) to rerank the results
    final_match = results.iloc[selected_indices, :].copy()

    return final_match

### Demo

In [7]:
rerank(search("companies who manufacture steel products", companies))

,Name,EmployeeCount,Description,Url,Region,Country,MetroArea,City,concatenated_name_description,concatenated_cleaned_text,embeddings,similarity,query_embeddings
0,United States Steel Corp.,10001+,United States Steel Corp. engages in the manuf...,ussteel.com,PA,US,Pittsburgh/PA Metro,Pittsburgh,United States Steel Corp. United States Steel ...,united states steel corp united states steel c...,"[-0.03792184963822365, 0.011723089963197708, -...",0.704631,"[[-0.09991994, -0.04503363, -0.013560827, 0.07..."
64,Boryszew SA,5001-10000,Boryszew SA engages in the production and sale...,boryszew.com.pl,MZ,PL,Warsaw/Poland Metro,Warsaw,Boryszew SA Boryszew SA engages in the product...,boryszew sa boryszew sa engages in the product...,"[-0.02254592441022396, -0.014332716353237629, ...",0.516604,"[[-0.09991994, -0.04503363, -0.013560827, 0.07..."
29,Takween Advanced Industries Co.,251-500,Takween Advanced Industries Co. engages in the...,takweenai.com,EP,SA,Riyadh/Saudi Arabia Metro,Khobar,Takween Advanced Industries Co. Takween Advanc...,takween advanced industries co takween advance...,"[-0.11291877925395966, -0.01817445456981659, -...",0.559428,"[[-0.09991994, -0.04503363, -0.013560827, 0.07..."
81,Macofil SA,101-250,Macofil SA engages in the manufacture and trad...,macofilsa.ro,GJ,RO,Bucharest/Romania Metro,Targu Jiu,Macofil SA Macofil SA engages in the manufactu...,macofil sa macofil sa engages in the manufactu...,"[-0.027906471863389015, -0.02500426024198532, ...",0.504042,"[[-0.09991994, -0.04503363, -0.013560827, 0.07..."
21,Insimbi Industrial Holdings Ltd.,501-1000,Insimbi Industrial Holdings Ltd. engages in th...,insimbi-group.co.za,GT,ZA,Johannesburg/S.Afr. Metro,Germiston,Insimbi Industrial Holdings Ltd. Insimbi Indus...,insimbi industrial holdings ltd insimbi indust...,"[-0.1477467268705368, 0.03411092236638069, -0....",0.577181,"[[-0.09991994, -0.04503363, -0.013560827, 0.07..."
38,Termovent SC Livnica Celika AD,101-250,Termovent SC Livnica Celika AD engages in the ...,livnica.com,NB,RS,Belgrade/Serbia Metro,Backa Topola,Termovent SC Livnica Celika AD Termovent SC Li...,termovent sc livnica celika ad termovent sc li...,"[0.006184963975101709, -0.04094167426228523, -...",0.546955,"[[-0.09991994, -0.04503363, -0.013560827, 0.07..."
55,"SIFCO Industries, Inc.",251-500,"SIFCO Industries, Inc. engages in the manufact...",sifco.com,OH,US,Cleveland/OH Metro,Cleveland,"SIFCO Industries, Inc. SIFCO Industries, Inc. ...",sifco industries inc sifco industries inc enga...,"[0.041652876883745193, -0.07092422991991043, -...",0.522987,"[[-0.09991994, -0.04503363, -0.013560827, 0.07..."
4,El Ezz Aldekhela Steel-Alexandria,1001-5000,El Ezz Aldekhela Steel-Alexandria engages in t...,NaN,GZ,EG,Africa (Northern) Metro,Giza,El Ezz Aldekhela Steel-Alexandria El Ezz Aldek...,el ezz aldekhela steelalexandria el ezz aldekh...,"[-0.10523804277181625, 0.03345298394560814, -0...",0.636608,"[[-0.09991994, -0.04503363, -0.013560827, 0.07..."
76,GrafTech International Ltd.,1001-5000,GrafTech International Ltd. engages in the man...,graftech.com,OH,US,Cleveland/OH Metro,Brooklyn Heights,GrafTech International Ltd. GrafTech Internati...,graftech international ltd graftech internatio...,"[-0.12607739865779877, -0.012034864164888859, ...",0.511285,"[[-0.09991994, -0.04503363, -0.013560827, 0.07..."
12,HG Metal Manufacturing Ltd.,101-250,HG Metal Manufacturing Ltd. is an investment h...,hgmetal.com,SW,SG,Singapore Metro,Singapore,HG Metal Manufacturing Ltd. HG Metal Manufactu...,hg metal manufacturing ltd hg metal manufactur...,"[-0.049937065690755844, 0.029586106538772583, ...",0.592543,"[[-0.09991994, -0.04503363, -0.013560827, 0.07..."


### Questions

#### 1. Breifly summarize your implementation

#### 2. What would be your next step(s) to increase performance?

#### 3. What would be your next step(s) to improve search quality?